In [ ]:
from demo import *

print("ml_analysis demo — end-to-end pipeline")

# --- Regenerate synthetic data ---
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

rng = np.random.default_rng(RANDOM_SEED)
assets = ["A01", "A02", "A03"]
classes = ["TP", "FP", "TN", "FN"]
replacement_types = ["bearing", "seal"]

print(f"\n[1/5] Generating synthetic data  ({len(assets)} assets × {len(classes)} classes"
        f" × {N_EVENTS_PER_CLASS} events each) ...")
labels = generate_synthetic_data(
    assets=assets,
    classes=classes,
    replacement_types=replacement_types,
    n_per_class=N_EVENTS_PER_CLASS,
    event_len_h=EVENT_LEN_HOURS,
    rng=rng,
)
print(f"   Label table: {labels.shape[0]} events")

print("\n[2/5] Registering features ...")
register_features()

print("\n[3/5] Building event dataset + materialising period aggregates ...")
events = build(labels, cfg=cfg)
period = to_period(
    events,
    cfg=cfg,
    aggregators=["mean", "std", "min", "max", "p05", "p95"],
)
print(f"   Period table: {period.shape[0]} rows × {period.shape[1]} columns")





In [ ]:
print("\n[4/5] Running analysis suite ...")
ctx = AnalysisContext(
    df=period,
    cfg=cfg,
    target_col="class",
    label_filter={"class": ["TP", "FP", "TN", "FN"]},
    stratify_by="replacement_type",
    output_dir=str(OUTPUT_DIR),
)

analyses = [
    DistributionAnalysis(),
    PairwiseSeparability(top_n=10),
    FeatureImportance(
        rf_params={"n_estimators": 200, "n_jobs": -1, "random_state": RANDOM_SEED},
        permutation_repeats=5,
    ),
    ClusterAnalysis(),
    ClassifierEvaluation(run_lgb=True, run_xgb=True),
    Stratified(
        inner=FeatureImportance(
            name="importance_strat",
            rf_params={"n_estimators": 100, "n_jobs": -1, "random_state": RANDOM_SEED},
            permutation_repeats=3,
        ),
        by="replacement_type",
    ),
]
results = run_analyses(analyses, ctx)

In [ ]:
cluster = results["clustering"]

print(f"Best k (KMeans): {cluster['best_k']}")
print(f"Algorithms fitted: {list(cluster['labels'])}")
print(f"Reductions available: {list(cluster['reductions'])}")
print(f"Class names: {cluster['class_names']}")

print_clustering(cluster)

In [ ]:
# 2-D projection coloured by true class vs. by KMeans assignment
import matplotlib.pyplot as plt
import numpy as np

reduction_name = "UMAP" if "UMAP" in cluster["reductions"] else "PCA"
emb = cluster["reductions"][reduction_name]
y_true = np.asarray([cluster["class_names"].index(c) if c in cluster["class_names"] else -1
                     for c in ctx.df[ctx.target_col].to_list()])
kmeans_key = next(k for k in cluster["labels"] if k.startswith("KMeans"))
y_km = cluster["labels"][kmeans_key]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (labels, title) in zip(
    axes,
    [(y_true, f"{reduction_name} — true class"),
     (y_km, f"{reduction_name} — {kmeans_key}")],
):
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10", s=18, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel(f"{reduction_name}-1"); ax.set_ylabel(f"{reduction_name}-2")
plt.tight_layout(); plt.show()

## Cluster Analysis

`ClusterAnalysis` runs **KMeans** (with auto-selected `k`), **DBSCAN**, and **HDBSCAN**
on the standardised feature matrix, then reduces to 2-D with **PCA** (and **UMAP**
if installed) for visualisation. Each clustering is scored against the true class
labels with silhouette / Davies-Bouldin (intrinsic) and ARI / NMI / V-measure
(alignment with the supervised labels).

In [ ]:
print("\n[5/5] Results")
print_distributions(results["distributions"])
print_pairwise(results["pairwise"])
print_importance(results["importance"])
print_clustering(results["clustering"])
print_classifier(results["classifier"])
print_stratified(results["stratified__importance_strat"])


save_summary_figure(
    importance_result=results["importance"],
    distributions_result=results["distributions"],
    output_dir=OUTPUT_DIR,
)

print("\nDone.\n")